<a href="https://colab.research.google.com/github/roblei007/Aula-Projeto1/blob/aprendendo-numpy/exercicios_aula_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3 - **Pandas:** refatorando o Sistema de Triagem (v3)

Curso: Técnicas de Programação (Python) — Trilha Analista de Dados

Na Aula 3 reescrevemos as análises usando NumPy. Ganhamos desempenho e capacidade analítica, mas a Seção 5 deixou claro um limite: filtrar **registros completos** (dicionários inteiros) por critério calculado ainda exigia um `zip` manual. NumPy é ótimo para colunas de números, mas desconfortável para dados mistos.

Pandas resolve isso. O banco — que até aqui era uma lista de dicionários — vira um `DataFrame`: uma estrutura tabular nativa para dados mistos. A maior parte das funções que escrevemos vira **uma linha**. E ganhamos capacidades novas (como `groupby`) que eram trabalhosas demais para fazer manualmente.

Nesta aula trabalhamos na branch `v3-pandas`. Ao final, fazemos o merge na `main`.

Na próxima aula, esse mesmo sistema vai operar sobre um dataset real do mercado de dados, com 10.000+ candidatos.

## Ponto de partida

A célula abaixo prepara o ambiente:

- Importa Pandas (`pd`) e NumPy (`np`)
- Disponibiliza as funções da v1 (caso você não tenha seu arquivo carregado)
- Cria o `banco_exemplo` com os 5 candidatos das aulas anteriores

In [1]:
import pandas as pd
import numpy as np

# Funções da v1 — fallback caso seu arquivo da aula anterior não esteja carregado
def classificar_nivel(anos):
    if anos < 0:  return "Inválido"
    if anos <= 2: return "Júnior"
    if anos <= 5: return "Pleno"
    return "Sênior"

def criar_candidato(nome, cargo, anos, pretensao, modalidade):
    return {
        "nome": nome, "cargo_desejado": cargo,
        "anos_experiencia": anos, "pretensao_salarial": pretensao,
        "modalidade": modalidade
    }

def adicionar_candidato(banco, candidato):
    return banco + [candidato]


# Banco de exemplo (mesmo das aulas anteriores)
banco_exemplo = []
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Ana Lima",     "Analista de Dados",   1, 55000,  "remoto"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Bruno Costa",  "Cientista de Dados",  4, 95000,  "híbrido"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Carla Mendes", "Engenheira de Dados", 8, 145000, "presencial"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Diego Souza",  "Analista de Dados",   2, 60000,  "remoto"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Elena Ferraz", "Cientista de Dados",  5, 100000, "híbrido"))

print(f"banco_exemplo: {len(banco_exemplo)} candidatos")

banco_exemplo: 5 candidatos


## 1. A grande mudança - converter o banco em DataFrame

**Problema nas v1 e v2:** o banco é uma lista de dicionários. Para acessar a coluna `pretensao_salarial`, precisamos sempre iterar. Para filtrar, mais iteração. Para agrupar, ainda mais.

**Conceito do Pandas:** o `DataFrame` é uma estrutura tabular nativa. Cada candidato é uma **linha**; cada atributo é uma **coluna**. Operações por coluna ou por linha são naturais.

```
pd.DataFrame(lista_de_dicts)   # cria DataFrame direto da lista
df['coluna']                    # acessa uma coluna como Series
df.shape                        # (linhas, colunas)
df.head()                       # primeiras 5 linhas
df['col'].apply(funcao)         # aplica função a cada elemento da coluna
```

Implemente `banco_para_dataframe(banco)` que:

1. Converte o banco em um `DataFrame`
2. Adiciona uma coluna `nivel` calculada a partir de `anos_experiencia` (use `classificar_nivel`)

**Exemplo:**

```
df = banco_para_dataframe(banco_exemplo)
df.shape       ->  (5, 6)
df.columns     ->  Index(['nome', 'cargo_desejado', 'anos_experiencia',
                          'pretensao_salarial', 'modalidade', 'nivel'])
df['nivel'][0] ->  'Júnior'
```

In [5]:
banco_exemplo

[{'nome': 'Ana Lima',
  'cargo_desejado': 'Analista de Dados',
  'anos_experiencia': 1,
  'pretensao_salarial': 55000,
  'modalidade': 'remoto'},
 {'nome': 'Bruno Costa',
  'cargo_desejado': 'Cientista de Dados',
  'anos_experiencia': 4,
  'pretensao_salarial': 95000,
  'modalidade': 'híbrido'},
 {'nome': 'Carla Mendes',
  'cargo_desejado': 'Engenheira de Dados',
  'anos_experiencia': 8,
  'pretensao_salarial': 145000,
  'modalidade': 'presencial'},
 {'nome': 'Diego Souza',
  'cargo_desejado': 'Analista de Dados',
  'anos_experiencia': 2,
  'pretensao_salarial': 60000,
  'modalidade': 'remoto'},
 {'nome': 'Elena Ferraz',
  'cargo_desejado': 'Cientista de Dados',
  'anos_experiencia': 5,
  'pretensao_salarial': 100000,
  'modalidade': 'híbrido'}]

In [4]:
pd.DataFrame(banco_exemplo)['anos_experiencia'].apply(classificar_nivel)


,anos_experiencia
0,Júnior
1,Pleno
2,Sênior
3,Júnior
4,Pleno


In [2]:
def banco_para_dataframe(banco):
    df = pd.DataFrame(banco)
    df['nivel'] = df['anos_experiencia'].apply(classificar_nivel)
    return df


In [6]:
# Testes
df = banco_para_dataframe(banco_exemplo)

print(isinstance(df, pd.DataFrame))
print(df.shape == (5, 6))
print('nivel' in df.columns)
print(df['nivel'].tolist() == ['Júnior', 'Pleno', 'Sênior', 'Júnior', 'Pleno'])

# Visualize o resultado
df

True
True
True
True


,nome,cargo_desejado,anos_experiencia,pretensao_salarial,modalidade,nivel
0,Ana Lima,Analista de Dados,1,55000,remoto,Júnior
1,Bruno Costa,Cientista de Dados,4,95000,híbrido,Pleno
2,Carla Mendes,Engenheira de Dados,8,145000,presencial,Sênior
3,Diego Souza,Analista de Dados,2,60000,remoto,Júnior
4,Elena Ferraz,Cientista de Dados,5,100000,híbrido,Pleno


**Sugestão de commit:**

```
feat: converte banco para DataFrame do Pandas

Substitui a representação lista-de-dicionarios por um DataFrame.
A coluna 'nivel' passa a ser materializada uma vez no momento da
conversao, em vez de calculada repetidamente nas funcoes de
filtro. Toda a v3 opera sobre DataFrames.
```

## 2. Filtros e ordenação — as funções viram uma linha



**Recapitulação:** na v1, filtrar candidatos por nível exigia um `for` com `if`. Em Pandas, uma comparação direta na coluna produz uma máscara booleana que pode ser usada para filtrar o DataFrame inteiro.

**Conceitos do Pandas:**

```
df[df['coluna'] == valor]                            # filtro por igualdade
df[(df['a'] >= x) & (df['b'] <= y)]                  # filtros combinados (use & e | , parenteses obrigatorios)
df.sort_values('coluna', ascending=False)            # ordenacao
```

Refatore as três funções abaixo. Todas aceitam um `DataFrame` (não mais um `banco`):

- `candidatos_por_nivel(df, nivel)` retorna o subconjunto do `df` com o nível informado
- `candidatos_na_faixa(df, salario_min, salario_max)` retorna candidatos cuja pretensão está na faixa
- `ranking_por_experiencia(df)` retorna o `df` ordenado por experiência, do maior para o menor

**Exemplo:**

```
candidatos_por_nivel(df, "Pleno")            # 2 candidatos (Bruno e Elena)
candidatos_na_faixa(df, 60000, 100000)        # 3 candidatos
ranking_por_experiencia(df).iloc[0]['nome']  # 'Carla Mendes'
```

In [7]:
df.sort_values('anos_experiencia', ascending=False)

,nome,cargo_desejado,anos_experiencia,pretensao_salarial,modalidade,nivel
2,Carla Mendes,Engenheira de Dados,8,145000,presencial,Sênior
4,Elena Ferraz,Cientista de Dados,5,100000,híbrido,Pleno
1,Bruno Costa,Cientista de Dados,4,95000,híbrido,Pleno
3,Diego Souza,Analista de Dados,2,60000,remoto,Júnior
0,Ana Lima,Analista de Dados,1,55000,remoto,Júnior


In [8]:
def candidatos_por_nivel(df, nivel):
    # sua solução aqui
    return df[df["nivel"] == nivel]

def candidatos_na_faixa(df, salario_min, salario_max):
    # sua solução aqui
    return df[(df['pretensao_salarial'] >= salario_min) & (df['pretensao_salarial'] <= salario_max)]

def ranking_por_experiencia(df):
    # sua solução aqui
    return df.sort_values('anos_experiencia', ascending=False)

In [9]:
# Testes
df = banco_para_dataframe(banco_exemplo)

plenos = candidatos_por_nivel(df, "Pleno")
print(len(plenos) == 2)
print(sorted(plenos['nome'].tolist()) == ['Bruno Costa', 'Elena Ferraz'])

na_faixa = candidatos_na_faixa(df, 60000, 100000)
print(len(na_faixa) == 3)

ranking = ranking_por_experiencia(df)
print(ranking.iloc[0]['nome']  == 'Carla Mendes')
print(ranking.iloc[-1]['nome'] == 'Ana Lima')

True
True
True
True
True


**Sugestão de commit:**

```
refactor: usa DataFrame para filtros e ordenacao

As tres funcoes (filtro por nivel, filtro por faixa, ranking)
passam de loops em Python puro para operacoes vetorizadas de
Pandas. Cada funcao caiu para uma unica expressao.
```

## 3. Estatísticas em uma linha - `describe()`



**Recapitulação:** na v2 escrevemos `estatisticas_completas` com 7 cálculos manuais via NumPy. Em Pandas, isso já existe pronto.

**Conceito do Pandas:**

```
df['coluna'].mean()       # media
df['coluna'].describe()   # estatisticas completas: count, mean, std, min, percentis, max
df.describe()             # describe sobre todas as colunas numericas
```

Implemente:

- `media_pretensao(df)` retorna a média da `pretensao_salarial`, arredondada para 2 casas
- `estatisticas_pretensao(df)` retorna o resultado de `.describe()` aplicado à coluna `pretensao_salarial`

**Exemplo:**

```
media_pretensao(df)        ->  91000.0
estatisticas_pretensao(df) ->  Series com count, mean, std, min, 25%, 50%, 75%, max
```

In [11]:
def media_pretensao(df):
    # sua solução aqui
    return round(df['pretensao_salarial'].mean(), 2)


def estatisticas_pretensao(df):
    # sua solução aqui
    return df['pretensao_salarial'].describe()

In [12]:
# Testes
df = banco_para_dataframe(banco_exemplo)

print(media_pretensao(df) == 91000.0)

stats = estatisticas_pretensao(df)
print(isinstance(stats, pd.Series))
print(stats['mean']  == 91000.0)
print(stats['min']   == 55000.0)
print(stats['max']   == 145000.0)
print(stats['count'] == 5.0)

# Inspecione o resultado completo
stats

True
True
True
True
True
True


,pretensao_salarial
count,5.000000
mean,91000.000000
std,36297.382826
min,55000.000000
25%,60000.000000
50%,95000.000000
75%,100000.000000
max,145000.000000


**Sugestão de commit:**

```
refactor: substitui estatisticas_completas por describe()

A funcao manual com 7 calculos da v2 vira uma chamada a
df['col'].describe(). Mesmo conteudo, codigo trivial, e
ainda ganhamos o count que nao tinhamos antes.
```

## 4. Análises agrupadas — `groupby`




**Capacidade nova:** até agora calculamos estatísticas sobre o banco inteiro. E se quisermos saber **a pretensão média por cargo desejado**? Ou **quantos candidatos há por nível**? Em Python puro, isso exigiria construir dicionários intermediários e iterar. Em Pandas, é uma linha.

**Conceito do Pandas:**

```
df.groupby('coluna_categoria')['coluna_numerica'].mean()    # media por grupo
df.groupby('coluna').size()                                 # tamanho de cada grupo
df['coluna'].value_counts()                                 # contagem por valor (idem ao acima, mais curto)
```

O resultado do `groupby` é uma `Series` indexada pelas categorias.

Implemente:

- `media_pretensao_por_cargo(df)` retorna a média da pretensão agrupada por `cargo_desejado`, arredondada para 2 casas
- `total_por_nivel(df)` retorna a contagem de candidatos por nível (use `groupby` + `size`)
- `distribuicao_por_modalidade(df)` retorna a contagem por modalidade (use `value_counts`)

**Exemplo:**

```
media_pretensao_por_cargo(df)
->  cargo_desejado
    Analista de Dados        57500.0
    Cientista de Dados       97500.0
    Engenheira de Dados     145000.0

total_por_nivel(df)
->  nivel
    Júnior    2
    Pleno     2
    Sênior    1
```

In [38]:
def media_pretensao_por_cargo(df):
    # sua solução aqui
   return df.groupby('cargo_desejado')['pretensao_salarial'].mean().round(2)


def total_por_nivel(df):
    # sua solução aqui
    return df.groupby('nivel').size()


def distribuicao_por_modalidade(df):
    # sua solução aqui
    return df.groupby('modalidade')['modalidade'].value_counts()


In [39]:
# Testes
df = banco_para_dataframe(banco_exemplo)

media_cargo = media_pretensao_por_cargo(df)
print(media_cargo['Analista de Dados']     == 57500.0)
print(media_cargo['Cientista de Dados']    == 97500.0)
print(media_cargo['Engenheira de Dados']   == 145000.0)

total_nivel = total_por_nivel(df)
print(total_nivel['Júnior'] == 2)
print(total_nivel['Pleno']  == 2)
print(total_nivel['Sênior'] == 1)

dist_mod = distribuicao_por_modalidade(df)
print(dist_mod['remoto']     == 2)
print(dist_mod['híbrido']    == 2)
print(dist_mod['presencial'] == 1)

True
True
True
True
True
True
True
True
True


In [40]:
df.groupby('cargo_desejado')['pretensao_salarial'].mean().round(2)

,pretensao_salarial
cargo_desejado,
Analista de Dados,57500.0
Cientista de Dados,97500.0
Engenheira de Dados,145000.0


**Sugestão de commit:**

```
feat: adiciona analises agrupadas (cargo, nivel, modalidade)

Tres analises que seriam dolorosas em Python puro viram
chamadas a groupby. Permite responder: qual cargo tem maior
pretensao salarial em media, quantos candidatos temos em cada
nivel, qual a distribuicao de modalidades de trabalho.
```

## 5. Reajuste salarial em DataFrame




**Recapitulação:** na v2, aplicar um reajuste exigia extrair o array de salários, multiplicar e reconstruir o banco com `zip`. Em Pandas, modificamos a coluna diretamente.

**Conceito do Pandas:**

```
df['coluna'] = df['coluna'] * fator       # atribuicao em coluna inteira
df.copy()                                  # copia para nao mutar o original
```

Implemente `aplicar_reajuste(df, percentual)` que retorna um **novo** `DataFrame` com a `pretensao_salarial` reajustada pelo percentual informado. O `df` original não deve ser alterado.

**Exemplo:**

```
df_novo = aplicar_reajuste(df, 10)
df_novo['pretensao_salarial'].iloc[0]   ->  60500.0   # 55000 * 1.10
df['pretensao_salarial'].iloc[0]        ->  55000     # original intacto
```

**Orientação:** comece com `df.copy()`. Sem isso, modificar a coluna do `df_novo` também modifica o original.

In [41]:
df.pretensao_salarial * 1.2

,pretensao_salarial
0,66000.0
1,114000.0
2,174000.0
3,72000.0
4,120000.0


In [42]:
def aplicar_reajuste(df, percentual):
    # sua solução aqui
    df_temp = df.copy()
    perc = 1+percentual/100
    df_temp['pretensao_salarial'] = df_temp['pretensao_salarial'] * perc

    return df_temp

In [43]:
# Testes
df = banco_para_dataframe(banco_exemplo)
df_novo = aplicar_reajuste(df, 10)

# Novo DataFrame tem salários reajustados (usa tolerância pequena para ponto flutuante)
print(abs(df_novo['pretensao_salarial'].iloc[0] - 60500.0)  < 0.01)
print(abs(df_novo['pretensao_salarial'].iloc[2] - 159500.0) < 0.01)

# Original permanece intacto
print(df['pretensao_salarial'].iloc[0] == 55000)
print(df['pretensao_salarial'].iloc[2] == 145000)

# Reajuste negativo também funciona
df_corte = aplicar_reajuste(df, -5)
print(abs(df_corte['pretensao_salarial'].iloc[0] - 52250.0) < 0.01)

True
True
True
True
True


**Sugestão de commit:**

```
refactor: aplica reajuste salarial diretamente no DataFrame

Substitui o zip-com-list-comprehension da v2 por atribuicao
direta em coluna. df.copy() preserva o DataFrame original,
mantendo a semantica funcional.
```

In [ ]:
## 6. Persistência: salvar e carregar em CSV




**Capacidade nova:** até aqui, o banco existia apenas em memória. Em qualquer trabalho real, os dados vêm de arquivos (CSV, Excel, banco de dados). Pandas lê e escreve esses formatos com uma linha.

**Conceitos do Pandas:**

```
df.to_csv('arquivo.csv', index=False)     # salva (index=False evita coluna extra de índice)
pd.read_csv('arquivo.csv')                # carrega
```

Implemente:

- `salvar_banco_csv(df, caminho)` salva o DataFrame em um arquivo CSV no caminho informado, sem coluna de índice
- `carregar_banco_csv(caminho)` carrega o CSV e retorna o DataFrame

**Exemplo:**

```
salvar_banco_csv(df, 'candidatos.csv')
df_carregado = carregar_banco_csv('candidatos.csv')
df_carregado.shape  ->  (5, 6)
```

In [50]:
def salvar_banco_csv(df, caminho):
    # sua solução aqui
    df.to_csv(caminho, index=False)
    return True


def carregar_banco_csv(caminho):
    # sua solução aqui
    return pd.read_csv(caminho)

In [51]:
# Testes — salva e recarrega
df = banco_para_dataframe(banco_exemplo)

salvar_banco_csv(df, 'candidatos.csv')
df_carregado = carregar_banco_csv('candidatos.csv')

print(df_carregado.shape == (5, 6))
print(df_carregado['nome'].tolist() == df['nome'].tolist())
print(df_carregado['pretensao_salarial'].sum() == df['pretensao_salarial'].sum())

# Inspecione
df_carregado

True
True
True


,nome,cargo_desejado,anos_experiencia,pretensao_salarial,modalidade,nivel
0,Ana Lima,Analista de Dados,1,55000,remoto,Júnior
1,Bruno Costa,Cientista de Dados,4,95000,híbrido,Pleno
2,Carla Mendes,Engenheira de Dados,8,145000,presencial,Sênior
3,Diego Souza,Analista de Dados,2,60000,remoto,Júnior
4,Elena Ferraz,Cientista de Dados,5,100000,híbrido,Pleno


**Sugestão de commit:**

```
feat: adiciona suporte a CSV para entrada e saida de dados

Permite persistir o banco em disco e carregar de volta. Em
qualquer cenario real de analise, os dados nao vem como
dicionarios Python: vem de arquivos. Esse e o primeiro passo
para a aula 5, onde o sistema vai operar sobre um dataset real
do mercado de dados.
```